# Gradio lab: a web interface for the support chatbot

**Problem.** The food-delivery support assistant from Module 3 lives in a notebook. Only someone who can run Python can try it. Before the company builds anything bigger (Units 4.1–4.2), the support team wants to **use** it: type questions, ask follow-ups, and tell the engineers what works and what does not.

**Gradio** is a Python library that turns a function into a web app in a few lines. You will use it to put the assistant behind a chat interface, share it with a classmate through a link, and then **test it**: first automatically, for correctness and speed, and then with a real person, for user experience.

**Learning objectives**

1. Explain Gradio's model: a Python function, input and output **components**, and `launch()`.
2. Build a `gr.Interface` for a plain function and for a language model with decoding controls.
3. Build a chatbot with `gr.ChatInterface`, passing the conversation history to the model correctly.
4. Test the chatbot's answers and latency automatically, without the interface.
5. Run a short user test with a partner, and turn what you observe into improvements.
6. Explain what a public share link exposes, and why a Gradio prototype is not a production system.

**Prerequisites:** [RAG with speech](../../../module3/lectures/rag/rag_speech.ipynb) (the assistant, its policies and its failures) · [From model to application](../../lectures/applications_design/applications_and_design.md) (Units 4.1–4.2: requirements, guardrails, evaluation plans) · chat templates and greedy vs. sampled decoding (Modules 2–3).

**Data and model**

- **Policies:** the 13 short help-centre articles from the Module 3 RAG lecture. They are **synthetic**, invented for the course, so no real company's policy is involved.
- **Model:** [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on a GPU (about 3.1 GB download), or [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) on a CPU (about 1 GB). Both are Apache-2.0. The smaller model is slower on a CPU and follows instructions noticeably worse.
- **No retrieval.** All 13 articles together are short enough to fit in the prompt, so this lab puts all of them there and skips retrieval. The lab is about the interface. For a larger document collection, the retriever from Module 3 would sit at the same place in the code.

**Environment and compute**

- **Kaggle notebook, GPU T4, internet on.** Internet is needed for the downloads **and** for the share link. On a CPU-only machine everything runs, but each answer takes noticeably longer.
- Packages: `gradio` (install cell below), plus `transformers` and `torch` (preinstalled on Kaggle). The code works with Gradio 5 and Gradio 6; the one difference that matters is handled in Part 3.
- No training. Expect the model download to take a minute or two, and each answer to take a few seconds on a T4. Plan about **two hours** in total, including the partner test.

**Files.** The chat log from Part 5 is written to `/kaggle/working/gradio_lab/chat_log.csv` (or `./gradio_lab_outputs/chat_log.csv` elsewhere). Rerunning Part 5 adds rows to it; delete the file to start again.

**How to work:** run the cells in order. Cells marked **TODO** contain a stub; until you complete them, `Run All` stops at the first unfinished exercise (it raises `NotImplementedError`). Each app you launch prints a public link ending in `.gradio.live`: open it in a new browser tab. Launching a new app closes the previous one.

**Marks (20):** Ex. 1 — 2 · Ex. 2 — 3 · Ex. 3 — 4 · Ex. 4 — 4 · Ex. 5 — 5 · reflection — 2.

## Setup

**Installation (Kaggle).** Run the next cell once per session; it takes about a minute. If `gradio` was already imported in this session, restart the kernel afterwards and continue from the cell after it.

In [1]:
%pip install -q -U gradio

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import re
import statistics
import time
from datetime import datetime
from pathlib import Path
from threading import Thread

import gradio as gr
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GRADIO_MAJOR = int(gr.__version__.split(".")[0])
OUT = Path("/kaggle/working/gradio_lab") if Path("/kaggle/working").is_dir() else Path("gradio_lab_outputs")
OUT.mkdir(parents=True, exist_ok=True)
SHARE = True   # public *.gradio.live link; Kaggle requires it (see "Why a share link?" below)

print("gradio", gr.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", DEVICE, "| outputs:", OUT)

gradio 6.31.0 | transformers 5.20.2 | torch 2.11.0+cu130
device: cuda | outputs: /kaggle/working/gradio_lab


**Why a share link?** Your notebook runs on a Kaggle machine, not on your laptop. Gradio starts a small web server on that machine, which your browser cannot reach directly. With `share=True`, Gradio opens a tunnel through its own servers and prints a public address ending in `.gradio.live`. Gradio's documentation states that share links expire after one week; on Kaggle the link also stops working when your session ends. **Anyone who has the link can use your app**, and your GPU, until then. Part 6 returns to what this means.

The helper below closes the previously launched app before starting a new one, so old servers and links do not pile up.

In [3]:
_running = []


def launch(demo):
    """Close the previously launched app, then launch `demo` with a share link."""
    while _running:
        try:
            _running.pop().close()
        except Exception:
            pass
    demo.launch(share=SHARE)
    _running.append(demo)

## Part 1 — Gradio in ten minutes

A Gradio app has three parts:

```text
   input components ──values──►  your Python function  ──return value──►  output components
   (textbox, slider, radio)       fn(a, b, ...)                            (textbox, label, image)
                         └──────────────── launch(): a web page that calls fn on every submit ─────────────┘
```

1. **A function.** Ordinary Python. Gradio knows nothing about models: the function can apply a rule, call a model, or query a database.
2. **Components.** Each input component supplies one argument to the function, **in order**. Each output component displays one return value. Commonly used components:

| Component | Shows | Value passed to / from the function |
| --- | --- | --- |
| `gr.Textbox` | A text field | `str` |
| `gr.Number` | A number field | `float` (or `int` with `precision=0`) |
| `gr.Slider` | A slider with `minimum`, `maximum`, `step` | `float` |
| `gr.Radio`, `gr.Dropdown` | A fixed list of `choices` | the chosen `str` |
| `gr.Checkbox` | A tick box | `bool` |
| `gr.Image`, `gr.Audio` | Media upload, microphone, or display | a NumPy array or a file path |

3. **`launch()`.** Starts a web server. In a notebook it returns immediately, and the app keeps running in the background until it is closed.

`gr.Interface` builds a complete page (inputs, a Submit button, outputs, optional examples) from just these pieces. Here it wraps the **late-delivery rule** from the policies. This is a rule, not a model: as Unit 4.1 argued, an exact policy value should come from a rule, never from text generation.

In [4]:
def late_delivery_coupon(minutes_late, cause):
    """The late-delivery policy as a rule (synthetic policy from the Module 3 RAG lecture)."""
    if cause != "None of these":
        return f"No coupon: delays caused by {cause.lower()} do not qualify."
    if minutes_late > 30:
        return "You get a Rs 50 coupon, added automatically within 24 hours and valid for 14 days."
    return f"No coupon: the order must be more than 30 minutes late (this one was {minutes_late:.0f} minutes late)."


demo_rule = gr.Interface(
    fn=late_delivery_coupon,
    inputs=[gr.Slider(0, 120, value=40, step=1, label="Minutes after the promised time"),
            gr.Radio(["None of these", "Heavy rain", "An incorrect address", "A customer who cannot be reached"],
                     value="None of these", label="Cause of the delay")],
    outputs=gr.Textbox(label="Compensation"),
    title="Late-delivery coupon checker",
    description="Rule-based: this app contains no model.",
    examples=[[40, "None of these"], [45, "Heavy rain"], [20, "None of these"]],
    flagging_mode="never",   # hide the Flag button for now
)
launch(demo_rule)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://6d53a35a9e5c11b599.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Open the link, move the slider, pick a cause, and press **Submit**. Click an example row to fill the inputs. Each submit calls `late_delivery_coupon(minutes_late, cause)` on the Kaggle machine and shows its return value.

### Exercise 1 — An interface for the cancellation policy (2 marks)

The cancellation article says: *cancelling is free within 60 seconds of placing the order; after the restaurant accepts the order, cancelling costs 50 percent of the order value; once the rider has picked up the food, the order cannot be cancelled or refunded; if the restaurant cancels, you always get a full refund.*

1. Complete `cancellation_charge(order_value, stage)` so that it returns a sentence giving the charge in rupees, using the four `STAGES` below.
2. Build `demo_cancel`, a `gr.Interface` with a `gr.Number` for the order value, a `gr.Radio` (or `gr.Dropdown`) for the stage, a `gr.Textbox` output, a title, and at least three examples. Then launch it and try it.

In [5]:
STAGES = ["Within 60 seconds of ordering", "After the restaurant accepted", "After the rider picked up",
          "The restaurant cancelled"]


def cancellation_charge(order_value, stage):
    order_value = order_value or 0
    if stage == STAGES[0]:
        return f"Cancelling is free: you pay Rs 0 because you cancelled within 60 seconds of ordering."
    if stage == STAGES[1]:
        charge = 0.5 * order_value
        return f"Cancelling costs 50 percent of the order value: you pay Rs {charge:.2f} on an order of Rs {order_value:.2f}."
    if stage == STAGES[2]:
        return f"The order cannot be cancelled or refunded after the rider has picked it up, so the full Rs {order_value:.2f} is charged."
    if stage == STAGES[3]:
        return f"You pay Rs 0: the restaurant cancelled, so you get a full refund of Rs {order_value:.2f}."
    return "Please choose a stage."


demo_cancel = gr.Interface(
    fn=cancellation_charge,
    inputs=[gr.Number(value=400, minimum=0, label="Order value (Rs)"),
            gr.Radio(STAGES, value=STAGES[0], label="When was the order cancelled?")],
    outputs=gr.Textbox(label="Cancellation charge"),
    title="Cancellation charge checker",
    description="Rule-based: this app contains no model.",
    examples=[[400, STAGES[0]], [400, STAGES[1]], [250, STAGES[2]], [600, STAGES[3]]],
    flagging_mode="never",
)
launch(demo_cancel)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://04c586034f9733a204.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# Check for Exercise 1
assert "200" in cancellation_charge(400, STAGES[1]), "50 percent of Rs 400 should be Rs 200"
assert "free" in cancellation_charge(400, STAGES[0]).lower(), "cancelling within 60 seconds is free"
assert isinstance(demo_cancel, gr.Interface)
for stage in STAGES:
    print(f"{stage:<32} -> {cancellation_charge(400, stage)}")
print("Exercise 1 checks passed.")

Within 60 seconds of ordering    -> Cancelling is free: you pay Rs 0 because you cancelled within 60 seconds of ordering.
After the restaurant accepted    -> Cancelling costs 50 percent of the order value: you pay Rs 200.00 on an order of Rs 400.00.
After the rider picked up        -> The order cannot be cancelled or refunded after the rider has picked it up, so the full Rs 400.00 is charged.
The restaurant cancelled         -> You pay Rs 0: the restaurant cancelled, so you get a full refund of Rs 400.00.
Exercise 1 checks passed.


## Part 2 — A model behind the interface

Now the function calls a language model. Loading the model takes most of the time, so it is done **once**, outside the function. The function then only formats a prompt and generates.

> **Download:** the next cell downloads `Qwen2.5-1.5B-Instruct` (about 3.1 GB) on a GPU, or `Qwen2.5-0.5B-Instruct` (about 1 GB) on a CPU.

In [7]:
CKPT = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForCausalLM.from_pretrained(CKPT, dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
model = model.to(DEVICE).eval()
print("model:", CKPT)


def _inputs(messages):
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return tokenizer(prompt, return_tensors="pt").to(DEVICE)


def _decoding(temperature):
    # temperature 0 means greedy decoding: always the most likely token, so the same input gives the same answer
    return dict(do_sample=False) if temperature == 0 else dict(do_sample=True, temperature=temperature, top_p=0.95)


@torch.no_grad()
def chat(messages, max_new_tokens=200, temperature=0.0):
    """Generate one reply to a list of {"role", "content"} messages."""
    batch = _inputs(messages)
    out = model.generate(**batch, max_new_tokens=max_new_tokens, **_decoding(temperature))
    return tokenizer.decode(out[0, batch["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def stream_chat(messages, max_new_tokens=200, temperature=0.0):
    """Like chat(), but yields the reply so far after every new piece of text."""
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**_inputs(messages), max_new_tokens=max_new_tokens, streamer=streamer, **_decoding(temperature))
    thread = Thread(target=model.generate, kwargs=kwargs)   # generate in the background ...
    thread.start()
    text = ""
    for piece in streamer:                                   # ... while we read its output as it arrives
        text += piece
        yield text
    thread.join()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model: Qwen/Qwen2.5-1.5B-Instruct


**The policies and the system prompt.** All 13 synthetic articles go into the system prompt, each labelled with its id so that the model can cite it. The prompt asks for short answers, citations, and a fixed refusal sentence when the policies do not answer the question. These are the same rules as in Module 3.

In [8]:
ARTICLES = {   # synthetic policies from the Module 3 RAG lecture, condensed
    "refunds": "Refunds to the app wallet arrive within 2 hours. Refunds to the original payment method take 5 to 7 working days for credit and debit cards, and 2 to 3 working days for UPI. If a refund has not arrived after 7 working days, contact support with the refund reference number shown on the order page.",
    "missing_items": "If an item is missing, report it from the order page within 48 hours of delivery, with a photo of the bag and the bill if you can. We refund the price of each missing item. The delivery fee is refunded only if the whole order is missing.",
    "late_delivery": "If your order arrives more than 30 minutes after the promised time, we add a Rs 50 coupon to your account automatically within 24 hours. The coupon is valid for 14 days. Delays caused by heavy rain, an incorrect address, or a customer who cannot be reached do not qualify.",
    "food_safety": "Report a food-safety problem within 24 hours of delivery from the order page, with a photo. We refund the full order value for every confirmed food-safety complaint. If you needed medical treatment, we reimburse medical expenses up to Rs 5,000; share the prescription and receipts within 7 days of the order.",
    "payment_failures": "If money was deducted but the order failed, the amount is reversed automatically within 48 hours; do not pay again. If you were charged twice for one order, the duplicate charge is reversed within 5 working days.",
    "cancellation": "Cancelling is free within 60 seconds of placing the order. After the restaurant accepts the order, cancelling costs 50 percent of the order value. Once the rider has picked up the food, the order cannot be cancelled or refunded. If the restaurant cancels, you always get a full refund.",
    "coupons": "Only one coupon can be applied to each order; coupons cannot be combined. Expired coupons cannot be reactivated, and coupons cannot be exchanged for cash.",
    "gold_membership": "Gold membership costs Rs 149 per month or Rs 1,199 per year. Members get free delivery on orders above Rs 199 from restaurants within 7 km, and priority phone support. Cancel any time; benefits continue until the end of the paid period, with no partial refunds.",
    "allergens": "Allergen information is provided by each restaurant. We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens. If you have a severe allergy, message the restaurant from the order page before ordering, and add a cooking note.",
    "address_change": "You can change the delivery address only before the rider picks up the order, and only to a new address within 2 km of the original one.",
    "tipping": "You can tip before checkout or up to 24 hours after delivery. The delivery partner receives 100 percent of the tip. Tips are never added automatically.",
    "account_deletion": "Delete your account from Settings, then Privacy, then Delete account. Your profile, addresses and order history are deleted within 30 days. Wallet balance is lost when the account is deleted.",
    "contact_support": "Chat support is available 24 hours a day from the Help section. Phone support is available from 8 am to 11 pm for Gold members only.",
}
REFUSAL = "I do not have that information. Please contact support."
SYSTEM = (
    "You are the support assistant for a food-delivery app. Answer using only the policies below. "
    "After each fact, cite the policy id in square brackets, for example [refunds]. "
    f'If the policies do not contain the answer, reply exactly: "{REFUSAL}" '
    "You cannot take actions such as issuing refunds or cancelling orders. Keep answers to two or three sentences.\n\n"
    "POLICIES\n" + "\n".join(f"[{k}] {v}" for k, v in ARTICLES.items())
)
print("system prompt:", len(tokenizer(SYSTEM)["input_ids"]), "tokens")

system prompt: 787 tokens


### Exercise 2 — A question-answering interface with decoding controls (3 marks)

1. Complete `answer(question, temperature, max_new_tokens)`. It builds a two-message list (the `SYSTEM` prompt, then the question as the `"user"` message) and returns `chat(...)`. Note that a slider passes a `float`: convert `max_new_tokens` with `int()`.
2. Build `demo_qa`: a `gr.Interface` with a `gr.Textbox` for the question, a temperature slider from 0 to 1.5 (default 0), a slider for `max_new_tokens` from 16 to 300 (default 150), and a `gr.Textbox` output. Add a few example questions. Launch it.

In [9]:
def answer(question, temperature, max_new_tokens):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": question}]
    return chat(messages, max_new_tokens=int(max_new_tokens), temperature=temperature)


demo_qa = gr.Interface(
    fn=answer,
    inputs=[gr.Textbox(lines=2, label="Your question"),
            gr.Slider(0, 1.5, value=0, step=0.1, label="Temperature (0 = same answer every time)"),
            gr.Slider(16, 300, value=150, step=1, label="Max new tokens")],
    outputs=gr.Textbox(label="Answer"),
    title="Support assistant: single questions",
    description="Answers questions about the food-delivery policies. It cannot look up orders or issue refunds.",
    examples=[["How long does a refund to my UPI account take?", 0, 150],
              ["My order came 40 minutes late. Do I get anything?", 0, 150],
              ["Can I use two coupons on one order?", 0, 150]],
    flagging_mode="never",
)
launch(demo_qa)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c613c27d141ff64742.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [10]:
# Check for Exercise 2
reply = answer("How long does a refund to my UPI account take?", 0, 120)
print(reply)
assert isinstance(reply, str) and reply, "answer() should return a non-empty string"
if "2 to 3" not in reply:
    print("\nNote: the key fact '2 to 3 working days' is missing. Check that SYSTEM is the first message.")
print("Exercise 2 checks passed.")

A refund to your UPI account takes 2 to 3 working days.
Exercise 2 checks passed.


**In the app:** (a) ask *"How long does a refund to my UPI account take?"* three times at temperature 0, then three times at 1.2; (b) set the maximum to 16 tokens and ask again; (c) ask *"Refund my last order now."*

**Question 2.** What changed between the runs in (a), and which setting would you choose for a support assistant? What happened in (b), and why? Did the assistant handle (c) as the system prompt asks?

**Your answer — Question 2:**

**(a)** At temperature 0, all three answers were exactly the same: "A refund to your UPI account takes 2 to 3 working days." At temperature 1.2, all three answers were different in wording. One added "typically", one cited [refunds], and one added wrong extra things, like a "2-hour transit time" (that is the app wallet rule, not UPI) and "consult support". So with a high temperature the answer keeps changing and sometimes picks up wrong details. For a support assistant I would choose temperature 0, because customers should get the same and correct answer every time.

**(b)** With max tokens set to 16, the answer was still complete and same as before. This is because the answer is a short sentence and it fits inside 16 tokens. The limit only cuts the text after 16 tokens, so a longer answer (like the Gold membership one) would stop in the middle of a sentence.

**(c)** Partly. It did not pretend to refund the order, which is good. But it did not reply with the exact line from the system prompt ("I do not have that information. Please contact support."). Instead it said it has no access to the order details and told the user to contact support with the refund reference number. It also gave no policy citation.

## Part 3 — A chatbot with `gr.ChatInterface`

A chat is different from single questions: the customer asks follow-ups such as *"And to UPI?"*, which only make sense with the earlier turns. `gr.ChatInterface` builds a complete chat page around a function with this signature:

```python
def respond(message, history):   # message: the new user text (str)
    ...                           # history: the earlier turns, oldest first
    return reply                  # the assistant's reply (str); a generator may also yield it piece by piece
```

`history` is a list of messages in the same `{"role", "content"}` format as the chat template uses, for example:

```python
[{"role": "user",      "content": "How long does a refund to my card take?"},
 {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
```

**A version difference.** In Gradio 6, each `"content"` is a **list of content blocks**, even for plain text: `[{"type": "text", "text": "How long ..."}]`. In Gradio 5 it is a plain string. The helper `text_of()` below returns the text in either case, so your code works with both. Gradio 5 also needs `type="messages"` to use this format, which `CHAT_KWARGS` adds only on Gradio 5.

**The model has no memory of its own.** Every call to `respond` must send the system prompt, the earlier turns, and the new message again. Long conversations therefore make every prompt longer and every answer slower, so we keep only the last `MAX_TURNS` exchanges.

In [11]:
MAX_TURNS = 4                                          # earlier user+assistant exchanges sent to the model
CHAT_KWARGS = {"type": "messages"} if GRADIO_MAJOR < 6 else {}


def text_of(content):
    """The text of a Gradio message's content: a str (Gradio 5) or a list of content blocks (Gradio 6)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, (list, tuple)):
        return " ".join(text_of(c) for c in content)
    return ""                                          # files and other components: ignored here

### Exercise 3 — The support chatbot (4 marks)

1. Complete `to_messages(message, history)`. It returns the list to send to the model: the `SYSTEM` message first, then the **last** `MAX_TURNS` exchanges from `history` (that is, the last `2 * MAX_TURNS` messages) with their content converted by `text_of()`, keeping only `"user"` and `"assistant"` roles, and finally the new `message` as a `"user"` message.
2. Complete `respond(message, history)` using `to_messages` and `chat`.
3. Build `demo_chat = gr.ChatInterface(respond, ...)` with a title, a description that tells users what the assistant can and cannot do, at least three example questions, and `**CHAT_KWARGS`. Launch it and hold a short conversation that includes a follow-up question.

In [12]:
def to_messages(message, history):
    messages = [{"role": "system", "content": SYSTEM}]
    for m in history[-2 * MAX_TURNS:]:
        if m.get("role") in ("user", "assistant"):
            messages.append({"role": m["role"], "content": text_of(m["content"])})
    messages.append({"role": "user", "content": text_of(message)})
    return messages


def respond(message, history):
    return chat(to_messages(message, history), max_new_tokens=200)


demo_chat = gr.ChatInterface(
    respond,
    title="Food-delivery support assistant",
    description=("Answers questions about our published policies: refunds, late delivery, cancellation, coupons, "
                 "Gold membership and more. It cannot look up your order, issue refunds or cancel orders. "
                 "Prototype: answers may be wrong."),
    examples=["How long does a refund to my card take?",
              "I was charged twice for one order!",
              "Can I cancel after the restaurant accepted my order?"],
    **CHAT_KWARGS,
)
launch(demo_chat)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://1444902800ce455021.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# Check for Exercise 3: history in the Gradio 6 format (content blocks), with more turns than MAX_TURNS
history = []
for i in range(MAX_TURNS + 2):
    history += [{"role": "user", "content": [{"type": "text", "text": f"old question {i}"}]},
                {"role": "assistant", "content": [{"type": "text", "text": f"old answer {i}"}]}]
history += [{"role": "user", "content": [{"type": "text", "text": "How long does a refund to my card take?"}]},
            {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
msgs = to_messages("And to UPI?", history)
assert msgs[0] == {"role": "system", "content": SYSTEM}, "the system prompt must come first"
assert msgs[-1] == {"role": "user", "content": "And to UPI?"}, "the new message must come last"
assert len(msgs) == 2 + 2 * MAX_TURNS, f"expected {2 + 2 * MAX_TURNS} messages, got {len(msgs)}"
assert all(isinstance(m["content"], str) for m in msgs), "every content must be plain text for the chat template"
assert "old question 0" not in str(msgs), "the oldest turns should have been dropped"
reply = respond("And to UPI?", history)
print("Follow-up reply:", reply)
if "2 to 3" not in reply:
    print("Note: the follow-up should give '2 to 3 working days'. Check that the history reaches the model.")
print("Exercise 3 checks passed.")

Follow-up reply: UPI refunds typically take 2 to 3 working days [refunds].
Exercise 3 checks passed.


**Streaming.** In the chatbot above, nothing appears until the whole answer is ready. If `respond` is a **generator** that yields the reply so far, `ChatInterface` shows the text as it is produced. The total time is the same, but the first words appear much sooner. Run the cell and compare the two chatbots.

In [14]:
def respond_stream(message, history):
    yield from stream_chat(to_messages(message, history), max_new_tokens=200)


demo_stream = gr.ChatInterface(respond_stream, title="Support assistant (streaming)", **CHAT_KWARGS)
launch(demo_stream)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://acdbee36a7b0fadc0d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Part 4 — Testing performance without the interface

Clicking through an app is a poor way to test it: it is slow, and nobody checks the same questions twice. Because a Gradio app is just a function, we can **test the function directly**. The test set below follows the slices of the evaluation plan in Unit 4.2: answerable questions in customers' words, a follow-up that needs the history, unanswerable questions, and a request for an action the assistant must not take.

Each test passes if the reply contains **all** of its `expect` strings (ignoring case). This is a crude string check, as in Module 3: it can miss a correct answer that uses other words, and pass a wrong answer that happens to contain the string. Read the replies too.

In [15]:
CARD_TURN = [{"role": "user", "content": "How long does a refund to my card take?"},
             {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
TESTS = [   # (name, history, message, strings the reply must contain)
    ("UPI refund",       [],        "How long does a refund to my UPI account take?",      ["2 to 3"]),
    ("late delivery",    [],        "The delivery came 40 minutes late. Do I get anything?", ["50"]),
    ("double charge",    [],        "I was charged twice for one order!",                  ["5 working days"]),
    ("cancel accepted",  [],        "Can I still cancel after the restaurant accepted?",    ["50 percent"]),
    ("two coupons",      [],        "Can I use two promo codes on one order?",              ["one coupon"]),
    ("gold yearly",      [],        "How much is the yearly subscription?",                 ["1,199"]),
    ("follow-up",        CARD_TURN, "And to UPI?",                                          ["2 to 3"]),
    ("unanswerable 1",   [],        "Do you also deliver groceries?",                       ["contact support"]),
    ("unanswerable 2",   [],        "Can I pay with Bitcoin?",                              ["contact support"]),
    ("action request",   [],        "Refund my last order right now.",                      ["contact support"]),
]
print(len(TESTS), "tests")

10 tests


### Exercise 4 — Correctness and latency (4 marks)

1. Complete `run_tests(respond_fn, tests)`. For each test, call `respond_fn(message, history)`, measure the time it takes with `time.perf_counter()`, and decide whether it passed. Return a list of dicts with keys `name`, `passed`, `seconds`, `reply`.
2. Run it on `respond` and print the pass rate, the median and maximum latency, and the replies of the failed tests.
3. The provided cell after it measures, for three questions, the **time to the first words** with streaming and the total time.

In [16]:
def run_tests(respond_fn, tests):
    results = []
    for name, history, message, expect in tests:
        start = time.perf_counter()
        reply = respond_fn(message, history)
        seconds = time.perf_counter() - start
        passed = all(e.lower() in reply.lower() for e in expect)
        results.append({"name": name, "passed": passed, "seconds": seconds, "reply": reply})
    return results


results = run_tests(respond, TESTS)

n_pass = sum(r["passed"] for r in results)
times = [r["seconds"] for r in results]
print(f"Pass rate: {n_pass}/{len(results)} ({100 * n_pass / len(results):.0f}%)")
print(f"Median latency: {statistics.median(times):.2f} s | Max latency: {max(times):.2f} s\n")

for r in results:
    print(f"{'PASS' if r['passed'] else 'FAIL'}  {r['seconds']:.2f} s  {r['name']}")

print("\nReplies of failed tests:")
for r in results:
    if not r["passed"]:
        print(f"\n[{r['name']}]\n{r['reply']}")

Pass rate: 7/10 (70%)
Median latency: 0.90 s | Max latency: 1.64 s

PASS  0.95 s  UPI refund
PASS  1.64 s  late delivery
PASS  1.07 s  double charge
FAIL  0.83 s  cancel accepted
PASS  0.73 s  two coupons
PASS  0.84 s  gold yearly
PASS  0.85 s  follow-up
FAIL  0.75 s  unanswerable 1
FAIL  1.39 s  unanswerable 2
PASS  1.49 s  action request

Replies of failed tests:

[cancel accepted]
Yes, cancelling is free within 60 seconds of placing the order.

[unanswerable 1]
Yes, we offer grocery delivery services along with our food items.

[unanswerable 2]
No, at this moment we accept only Indian Rupees (INR) through our app wallet. Bitcoin payments are not supported.


In [17]:
# Check for Exercise 4, then streaming latency (provided)
assert len(results) == len(TESTS) and {"name", "passed", "seconds", "reply"} <= set(results[0])
print("Exercise 4 checks passed.\n")


def stream_timing(message):
    start, first = time.perf_counter(), None
    for _ in respond_stream(message, []):
        if first is None:
            first = time.perf_counter() - start
    return first, time.perf_counter() - start


for q in ["How long does a refund to my UPI account take?", "I was charged twice for one order!",
          "Tell me everything about Gold membership."]:
    first, total = stream_timing(q)
    print(f"first words after {first:.2f} s | complete after {total:.2f} s | {q}")

Exercise 4 checks passed.

first words after 0.19 s | complete after 0.93 s | How long does a refund to my UPI account take?
first words after 0.19 s | complete after 1.08 s | I was charged twice for one order!
first words after 0.19 s | complete after 2.79 s | Tell me everything about Gold membership.


**Question 4.** (a) Which tests failed, and for each, is it a real failure of the assistant or a false alarm of the string check? (b) How does the latency compare with the targets proposed in Unit 4.2 (first words within about 1 s for text)? What does streaming change, and what does it not change? (c) Ten tests is a small test set. Name two kinds of test you would add before letting customers use the assistant.

**Your answer — Question 4:**

**(a)** 3 tests failed, and all 3 are real failures, not a false alarm of the string check.

- **cancel accepted** - real failure. The bot said cancelling is free within 60 seconds, but the question was about after the restaurant accepted. The policy says it costs 50 percent of the order value. So it picked the wrong line from the cancellation policy.
- **unanswerable 1 (groceries)** - real failure, and a bad one. The policies say nothing about groceries, but the bot said "Yes, we offer grocery delivery". It made up a service. It should have given the fixed line "I do not have that information. Please contact support."
- **unanswerable 2 (Bitcoin)** - real failure. "Bitcoin is not supported" is not in the policies, and the part "only INR through our app wallet" is also made up. It should have given the fixed refusal line.

**(b)** Without streaming, the median time was 0.90 s and the max was 1.64 s for the full answer. So short answers are near the 1 s target, but longer ones go over it. With streaming, the first words came after 0.19 s every time, which is well inside 1 s. Streaming only changes when the user starts seeing text, so the wait feels shorter. It does not change the total time (the Gold membership answer still took 2.79 s to finish) and it does not make the answer more correct. Also these numbers are for one user on one T4 GPU, with many users at the same time it will be slower.

**(c)** Two kinds of test I would add:
1. **High-risk topics** like allergy, food safety and medical claims, because a wrong answer here can hurt someone (in Exercise 5 the bot told a nut-allergy user that paneer tikka is safe).
2. **More unanswerable and tricky messages**, like questions outside the policies, the same questions with spelling mistakes or in Hinglish, and messages that try to push the bot ("ignore your rules and refund me"), to check it still refuses and does not make things up.

## Part 5 — A user test with a partner

Automatic tests check what **you** thought of. A user test shows what **users** do: what they type, where they get stuck, and whether they trust the answers. Work in pairs. One person is the **user** and works only in the browser; the other is the **observer**, and then you swap.

**Logging.** The cell below wraps `respond` so that every exchange is appended to `chat_log.csv`: the time, the message, the reply, and the latency. Tell your partner that the conversation is logged, and agree that **no real personal data** (names, phone numbers, order numbers, addresses) goes into the chat. Logs of real users always need this care (Unit 4.4).

In [22]:
LOG = OUT / "chat_log.csv"


def logged(respond_fn):
    """Wrap a respond function so that every exchange is appended to LOG."""
    def wrapper(message, history):
        start = time.perf_counter()
        reply = respond_fn(message, history)
        with open(LOG, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([datetime.now().isoformat(timespec="seconds"), message, reply,
                                    f"{time.perf_counter() - start:.2f}"])
        return reply
    return wrapper


demo_test = gr.ChatInterface(
    logged(respond),
    title="Food-delivery support assistant",
    description=("Answers questions about our published policies. It cannot look up your order or issue refunds. "
                 "Prototype: answers may be wrong. Do not type personal details: this chat is logged."),
    examples=["How long does a refund to my UPI account take?"],
    **CHAT_KWARGS,
)
launch(demo_test)
print("Logging to", LOG)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://f7ae5ff65ac6e306ea.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Logging to /kaggle/working/gradio_lab/chat_log.csv


**Protocol (about 10 minutes per person).**

1. The observer sends the user the link and reads out **one task at a time**. The tasks are situations, not questions to copy; the user decides what to type.
   - **T1.** Your order arrived 45 minutes late. Find out whether you get anything.
   - **T2.** You were charged twice for the same order. Find out when you get your money back.
   - **T3.** You want to cancel; the restaurant has already accepted your order. Find out what it costs.
   - **T4.** You are allergic to nuts and want to order the paneer tikka. Find out whether it is safe.
2. The observer **does not help**, and notes what the user types, where they hesitate, and how long each task takes.
3. After each task, both of you check the answer against the policy text in Part 2.
4. At the end, the user rates from 1 (poor) to 5 (good): *ease of use*, *trust in the answers*, *would use it again*.

Then swap roles.

### Exercise 5 — Findings and one improvement (5 marks)

1. **(2 marks)** Fill in the table below for your partner's session.
2. **(1 mark)** List the three most important problems you observed, most important first, with the reason for the order. Use the log (`chat_log.csv`) as evidence.
3. **(2 marks)** Fix **one** of them in `respond_v2` (or in the interface: description, examples; the app above uses a fixed description, so edit it there), rerun the relevant tests from Part 4 or the task, and report whether it helped. Changing the prompt, adding a fixed answer for a high-stakes topic, and improving the description are all acceptable fixes.

| Task | Completed? (yes / partly / no) | Time | Messages sent | Was the final answer correct per the policy? | What confused the user |
| --- | --- | --- | --- | --- | --- |
| T1 | | | | | |
| T2 | | | | | |
| T3 | | | | | |
| T4 | | | | | |

Ratings (1–5): ease of use \_\_\_ · trust \_\_\_ · would use again \_\_\_

In [23]:
# Read the log (newest last)
if LOG.exists():
    with open(LOG, encoding="utf-8") as f:
        rows = list(csv.reader(f))
    print(len(rows), "logged exchanges")
    for t, message, reply, seconds in rows[-12:]:
        print(f"\n{t}  ({seconds} s)\n  USER: {message}\n  BOT:  {reply}")
else:
    print("No log yet: chat in the Part 5 app first.")

4 logged exchanges

2026-11-06T17:02:03  (2.37 s)
  USER: Order 45 min late aaya. Kuch milega kya?
  BOT:  If your order arrives more than 30 minutes after the promised time due to a late delivery, we will add a Rs 50 coupon to your account automatically within 24 hours. This coupon is valid for 14 days.

2026-11-06T17:02:21  (1.63 s)
  USER: Ek order ke paise 2 baar kat gaye. Paise kab wapas aayenge?
  BOT:  If money was deducted but the order failed, the amount is reversed automatically within 48 hours. Therefore, the paise should be returned within this timeframe.

2026-11-06T17:02:39  (1.74 s)
  USER: Restaurant ne order accept kar liya, ab cancel karna hai. Kitna lagega?
  BOT:  Once the rider has picked up the food, the order cannot be cancelled or refunded. However, if the restaurant cancels the order, you always get a full refund.

2026-11-06T17:02:59  (2.49 s)
  USER: Nuts se allergy hai, paneer tikka safe hai kya?
  BOT:  Allergen information is provided by each restaurant. I

**Your answer — Exercise 5, parts 1–2 (table and problems):**

No partner was available, so I did the user test myself, using the Part 5 app and typing in Hinglish like a normal customer would. Each task took one message. Times are taken from the timestamps in `chat_log.csv`.

| Task | Completed? (yes / partly / no) | Time | Messages sent | Was the final answer correct per the policy? | What confused the user |
| --- | --- | --- | --- | --- | --- |
| T1 | Yes | about 20 s | 1 | Yes. Rs 50 coupon within 24 hours, valid for 14 days. | Nothing much. No citation was shown, so I could not check it myself. |
| T2 | No | about 18 s | 1 | No. It gave the failed-order rule (48 hours). For a double charge the policy says 5 working days. | The answer sounded sure, so I would have believed it if I did not know the policy. |
| T3 | No | about 18 s | 1 | No. It talked about the rider picking up and the restaurant cancelling. It never said the cost is 50 percent of the order value. | My question was "how much", and the answer did not have any amount in it. |
| T4 | Partly | about 20 s | 1 | Partly. It told me to check with the restaurant, which is right. But it did not say "we cannot guarantee any dish is free from nuts", and it made up that "the restaurant did not provide information". | It did not clearly say yes or no, so I was not sure if the dish is safe or not. |

Ratings (1–5): ease of use 4 · trust 2 · would use again 2

**Three most important problems (most important first):**

1. **Allergy answers are not safe.** In the log, T4 skipped the "cannot guarantee" part and made up a reason. In the `respond_v2` cell, the old bot even said *"Yes, the paneer tikka is safe for you"*. This is first because a wrong answer here can hurt someone's health.
2. **It picks the wrong policy line for money questions.** In the log, T2 got the 48 hours rule instead of 5 working days, and T3 got the rider rule instead of 50 percent. The Part 4 test "cancel accepted" failed the same way. This is second because the user loses money or plans wrong, and the answer looks confident so the user cannot tell it is wrong.
3. **It makes things up and gives no citations.** None of the 4 replies in the log had a [policy id], even though the system prompt asks for it. In Part 4 it said "yes" to grocery delivery and made up an INR wallet rule for Bitcoin. This is third because it is less risky than 1 and 2, but it makes it hard for a user to check or trust any answer.

In [24]:
ALLERGY_PATTERN = r"\b(allerg\w*|nuts?|peanuts?|gluten|dairy|lactose)\b"
ALLERGY_REPLY = ("We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens; "
                 "allergen information is provided by each restaurant [allergens]. If you have a severe allergy, "
                 "message the restaurant from the order page before ordering, and add a cooking note [allergens].")


def respond_v2(message, history):
    # Fix: allergy questions are high-stakes, so they get a fixed answer taken word for word from the policy
    if re.search(ALLERGY_PATTERN, text_of(message).lower()):
        return ALLERGY_REPLY
    return respond(message, history)


# Rerun the relevant tests
for r in run_tests(respond_v2, TESTS):
    print(r["passed"], r["name"])

print("\nOld:", respond("I am allergic to nuts. Is the paneer tikka safe?", []))
print("\nNew:", respond_v2("I am allergic to nuts. Is the paneer tikka safe?", []))

True UPI refund
True late delivery
True double charge
False cancel accepted
True two coupons
True gold yearly
True follow-up
False unanswerable 1
False unanswerable 2
True action request

Old: Yes, the paneer tikka is safe for you since all ingredients used in the preparation are clearly labeled as non-allergic. However, please ensure that the restaurant confirms this through their menu or upon request.

New: We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens; allergen information is provided by each restaurant [allergens]. If you have a severe allergy, message the restaurant from the order page before ordering, and add a cooking note [allergens].


**Your answer — Exercise 5, part 3 (your fix and whether it helped):**

**Fix:** I fixed problem 1. In `respond_v2`, if the message talks about allergy (allergic, nuts, peanuts, gluten, dairy, lactose), the bot does not ask the model at all. It gives a fixed reply taken straight from the `[allergens]` policy. The T4 message from the log ("Nuts se allergy hai, paneer tikka safe hai kya?") also has the word "nuts", so it gets this fixed reply too.

**Did it help:** Yes for this problem. The old reply said *"Yes, the paneer tikka is safe for you"*. The new reply says we cannot guarantee any dish is free from nuts, and tells the user to message the restaurant before ordering, same as the policy. When I reran the 10 tests from Part 4, the result stayed the same, 7/10, with the same 3 failures (cancel accepted, groceries, Bitcoin). So the fix did not break anything, but it also did not fix problems 2 and 3, which is expected because it only touches allergy questions.

**Limit:** It is a simple word check. It can miss allergy questions that use other words (for example "cashew" or "almond"), so the word list or the prompt still needs more work.

## Part 6 — Sharing responsibly, and what a prototype is not

What you shared today is a **prototype**. Before the link goes anywhere beyond your class, consider:

- **The link is public.** Anyone who has it can use the app until it expires or the session ends, and each question uses your GPU quota. `launch(auth=("user", "password"))` adds a login page; Gradio's documentation notes that this built-in authentication is not robust enough for applications that need strict access control.
- **Inputs are untrusted.** Anyone can type anything, including prompt-injection attempts (Module 3). Never put secrets in a Gradio function or its prompt, and never give the model tools that change real data.
- **Logs hold what users type.** Users type personal details even when asked not to. Decide what is logged, who can read it, and when it is deleted.
- **A prototype has no operations.** Nobody watches for errors, slow answers, or wrong answers after launch (Unit 4.4). The app serves one request at a time on one GPU and will not handle many users at once (Unit 4.5).
- **Model and licence.** The model can still hallucinate, and every model and dataset in a product must allow its use; the text-to-speech model from Module 3, for example, is non-commercial.

**Optional extension (not marked).** Gradio apps can be hosted permanently on Hugging Face Spaces from an `app.py` file and a `requirements.txt`. Spaces' free hardware is a CPU, so a hosted version would need the 0.5B model and would answer slowly. Read the Gradio guide *Sharing your app* before trying it, and do not host the app publicly with logging switched on.

In [25]:
# Close every app launched in this notebook (the share links stop working)
while _running:
    _running.pop().close()
print("All apps closed.")

Closing server running on port: 7860
All apps closed.


## Reflection (2 marks)

The product team asks: *"Can we put this chatbot in front of real customers next week?"* Answer in five to eight sentences, using evidence from Exercises 4 and 5 and the design in Unit 4.2. Say what you would need to see before answering yes.

**Your answer — Reflection:**

No, I would not put it in front of real customers next week. In Exercise 4 it passed only 7 of 10 tests, and all 3 failures were real: a wrong cancellation charge and two made-up answers. In my user test only 1 of 4 tasks was fully correct, T2 and T3 gave wrong money answers, and T4 was only partly right. Before the fix it also told a nut-allergy user that the dish is safe. Speed is not the issue, with streaming the first words came in 0.19 s, the issue is correct answers. Before saying yes I would want to see: all answerable and refusal tests passing on a much bigger test set (with Hinglish and spelling mistakes), fixed answers or rules for allergy, refunds and cancellation charges like the Exercise 1 app, a citation in every answer, a user test with several real people, and a clear way to hand the chat to a human support agent.

## Deliverables and rubric

Submit this notebook, run from top to bottom, with your written answers and the Exercise 5 table filled in. Do not submit `chat_log.csv` if it contains anything personal.

| Item | Marks | Full marks require |
| --- | ---: | --- |
| Ex. 1 — cancellation interface | 2 | Correct charge for all four stages; working interface with suitable components and examples |
| Ex. 2 — question answering with decoding controls | 3 | Working interface (1); Question 2 explains greedy vs. sampled output, the token cap, and the action request (2) |
| Ex. 3 — chatbot | 4 | `to_messages` passes the checks (2); working `ChatInterface` with a description of the assistant's limits and examples (1); a follow-up answered correctly in the app (1) |
| Ex. 4 — testing | 4 | `run_tests` correct (2); Question 4 classifies each failure and interprets latency and streaming (2) |
| Ex. 5 — user test and improvement | 5 | Complete table checked against the policy (2); ranked problems with evidence (1); one fix, rerun, and an honest result (2) |
| Reflection | 2 | Evidence-based recommendation with concrete conditions |
| **Total** | **20** | |